# Differentiable Nonlinear-MPC for Autonomous Vehicle in JAX

![NLP Drift](Image/NLP_Drift.png)


 
Since, Automated drifting represents an extreme control regime in which the vehicle is deliberately driven into highly nonlinear and unstable operating regions characterized by large sideslip angles and tire saturation. In such conditions, the underlying assumptions of linear or mildly nonlinear models break down, and traditional MPC pipelines, typically based on repeated linearization and quadratic programming (QP), become computationally inefficient and potentially unreliable[11]. 

Parallel to this, recent developments in Nonlinear Model Predictive Control (NMPC) demonstrate that integrating motion planning, tracking, and stability into a single nonlinear optimization problem significantly improves performance in high-speed obstacle avoidance scenarios[11]. However, such approaches still rely on approximations to achieve real-time feasibility, often sacrificing model fidelity in highly coupled dynamic regimes. 

Another critical limitation arises from model uncertainty, i.e., real-world systems rarely conform to a single accurate model. The comparative study of multiple-model algorithms achieves strong robustness by fusing predictions from multiple candidate models, particularly under unknown or rapidly changing dynamics. This suggests NMPC becomes substantially harder when the controlled domain is intentionally operated beyond typical linear/near-equilibrium regimes. 

Automated drifting can be considered precisely such a domain as it requires maintaining high sideslip angles, exploiting nonlinear tire forces, and keeping rear tires saturated while still meeting trajectory objectives. Therefore, in general, drifting regime forces simultaneous engagement with four coupled difficulties. 1) Nonlinearities and strong coupling in which the tire force response is significantly nonlinear and coupled; maintaining drift depends on balancing front lateral forces (often within friction limits) against rear forces (beyond friction limits), with longitudinal slip at the rear playing a central role. 2) Unstable equilibria in which high sideslip drift equilibria can be unstable and the controller must regulate around equilibria rather than simply track a benign reference. 3) Tight actuator and rate constraints, which focuses that drifting requires aggressive steering and torque modulation, but real vehicles impose limits such as (e.g. steering torque/angle/rate/drivetrain dynamics). 4) Real-time computational constraints: MPC must solve an optimization problem every sample, more importantly, insisting on exact solutions is often impractical and sometimes undesirable due to computation delays and model mismatch, instead structure-exploiting approximate solutions can yield excellent closed-loop performance[9].


At the same time, advances in differentiable programming have introduced a paradigm shift in how optimization and computation are structured. Instead of treating derivatives as auxiliary computations, differentiable programming treats entire algorithms as composable, differentiable computational graphs. This enables efficient gradient computation, higher-order derivatives, and parallel execution through modern frameworks such as JAX, which provide composable transformations, automatic differentiation, JIT compilation via XLA, and vectorization that can be combined arbitrarily. The key opportunity emerging from these developments is to rethink NMPC as a compiled differentiable program instead of a sequence of optimization problems solved iteratively. Many components of NMPC such as trajectory simulation rollouts, cost accumulation, Jacobian/Hessian computations, Gauss–Newton approximations, KKT solves, and constraint evaluation are inherently differentiable and can be restructured into a unified computational graph. This restructuring allows [5]:

1). Efficient gradient computation via automatic differentiation.  
2). Parallel evaluation of multiple candidate trajectories  
3). Provide on-demand derivatives/HVPs with predictable complexity/memory trade-offs including checkpointing strategies.  

Therefore, building upon these insights, this project proposes a novel control framework, which integrates three key strategies that are physically meaningful.  
1) Differentiable Programming JAX-based NMPC: Recasting the NMPC problem as a fully differentiable program for efficient gradient-based optimization.  
2) Multi-model probabilistic fusion: Incorporating multiple dynamic models to handle uncertainty and model mismatch robustly to severe nonlinearities and saturation.  
3) Parallelized optimization architecture: Leveraging vectorization and just-in-time (JIT) compilation to achieve real-time performance based on differentiable programming and semi-decentralized coordination concepts.  



## Literature Review ## 

This literature review is organized around algorithmic structure that is basically  based on how optimization is actually performed and how is that structure with respect to nonlinearity, constraints, and real-time feasibility.  

A widely deployed NMPC implementation pattern is: 1) Choose a transcription (single shooting, multiple shooting, collocation), which then solves the resulting finite-dimensional NLP at each time step using SQP or interior-point methods, 2) apply the first control input and shift the horizon (receding horizon). And then MPC OCP is solved repeatedly and approximately; therefore, key errors stem from discretization and inexact optimization, and algorithmic choices must exploit OCP structure.  


Direct multiple shooting is also particularly important because it converts the continuous-time OCP into an NLP with explicit defect constraints linking shooting nodes, in which it resembles the discrete-time OCP form and has favorable numerical properties for unstable dynamics. This is directly relevant to drifting, where the dynamics around equilibria can be unstable and where convergence from poor guesses is a primary risk.  

SQP with Gauss–Newton structure is a classic route where the linearized dynamics/constraints approximate the Hessian (often Gauss–Newton), solve a QP, and iterate. The drifting NMPC cases use the ACADO toolkit with an SQP solver in real time.  

Therefore, across NMPC implementation, runtime is dominated by:  

1) Evaluating dynamics and constraints along the horizon  
2) Computing Jacobians and approximate Hessians  
3) Solving a structured linear system (QP/KKT) efficiently  

Therefore, differentiable programming literature [4] quantifies complexity/memory regimes, in which reverse-mode autodiff can compute gradients efficiently particularly for scalar outputs but requires storing intermediate computation, checkpointing trades recomputation for memory. These trade-offs matter directly for NMPC rollouts over horizons.  

Furthermore, drifting control is inherently sensitive to uncertainty. For instance, friction variations, tire parameter mismatch, and disturbances change equilibrium and can destabilize high sideslip operation. Among others, robust MPC literature provides two key tools:  
 
1) Tube MPC and constraint tightening: Tube MPC bounds deviations between nominal and actual trajectories using invariant sets, homothetic tube MPC introduces scaling variables $\alpha$ to reduce conservatism relative to rigid tubes. This is relevant to drifting because feasible sets near saturation can be narrow; naive tightening may eliminate feasible drift.  

2) Stochastic MPC and chance constraints: Stochastic uncertainty, especially multiplicative effects complicates prediction and constraint satisfaction. Stochastic MPC frameworks address   constraints via sampling/scenario approaches. For drifting, friction uncertainty behaves closer to multiplicative uncertainty on forces. A drifting NMPC pipeline should therefore integrate uncertainty either via robust tubes, stochastic models, or scenario-based validation.  

3) Distributed optimization and control: Modern MPC increasingly engages distributed optimization (ADMM, ALADIN) and scalable architectures. This becomes highly relevant if drifting control is extended to multi-vehicle scenarios, or if within-vehicle control is decomposed across actuators and subsystems (steering, powertrain, brake-based yaw moment, etc.) [6][7].  


The semi-decentralized multi-agent framework introduces a key abstraction with respect to probabilistic information flow via selector infrastructure that can toggle which observations/actions are stored in centralized vs. decentralized memories, defining mixed policies that interpolate between full decentralization and full centralization [8]. The RS-SDA* algorithm shows how this interpolation can preserve much of the value of centralized coordination while improving tractability and coping communication limitations. For NMPC, this motivates a semi-decentralized optimization architecture in which some decision variables (e.g. steering trajectory) are optimized centrally at high rate, others (e.g. wheel torque distribution, thermal constraints, actuator dynamics) are optimized locally with intermittent coupling updates and communication between modules is probabilistic or delayed, and the control algorithm is explicitly designed to remain stable under such information constraints. This concept aligns with real vehicle software architectures, where different ECUs operate at different rates and with constrained interfaces, which is exactly the difficulty encountered in the drifting NMPC experiments where steering torque availability through standard vehicle interfaces limited full automation [9].  

Therefore, from the literature, key gaps identified are:  
1) Non-convexity handling remains under-structured, where classical NMPC solvers often assume warm start + local convergence is enough. Drifting can invalidate this due to multiple equilibria and saturation-induced sensitivity.  
2) Derivative computation is treated as an implementation detail, not an algorithmic design axis; however, in the philosophy of differentiable programming, it argues the opposite, which is that program design and differentiation are central.  
3) Distributed/semi-decentralized control ideas are underused in vehicle NMPC, despite clear architectural relevance to real ECUs and actuator constraints.  
4) Validation is not integrated as a first-class driver of the solver design (e.g. scenario generation that targets solver failure modes, or differentiable sensi tivity analysis to expose brittle constraints).  

Key computational bottlenecks and solver limitations. 

1) SQP requires repeated QP solves, and near saturation, linearization can be poor an require trust regions/line search. Drifting is precisely where linearization could be risky as equilibrium and non uniqueness implies multiple local optima , and single warm start SQP can converge to suboptimal equilbria or fail.
2) Simplified front tire model and neglected load transfer can shift equilibrium feasibility, and the equailibriym map may no longer match reality.
 Therefore, these limitation define the opportunity space for a new solver paradigm where convergence can be improved through multi start capability as well as lowering the per iteration cost can be exploited with parallelism. Thus, the central idea is to reinterepret NMPC as a differentiable program whose compilation and differention are part of the solver design. 




## Reconstructing the drifting NMPC baseline: 

JAX provides:
* Automatic differentiation of Python/NumPy functions (including higher-order derivatives), composable with control flow. 
* JIT compilation of pure functions using XLA, enabling end-to-end compilation and fusion. 
* Vectorization via vmap, pushing loops into primitives for performance and enabling parallel evaluation of multiple candidates.

The differentiable programming perspective emphasizes that programs must be designed for differentiation, and that differentiation through optimization and integration is central. 
In NMPC, the “program” is the composition:

$$controls u ↦ rollout_{x0:N (u)} ↦ cost_{J(u)} ↦ optimizer update.$$
The proposed new design axis is: choose a program structure that makes derivatives cheap and stable, and compile it. That is, 
 - Start with control inputs u
 - Simulate system dynamics to get trajectory x
 - Evaluate a cost function J(u)
 - Update controls using an optimizer


## Mathematical framework: 
 Let the NMPC decision variables be $w = (x, u)$, where  
$x = (x_0, \ldots, x_N)$,  
$u = (u_0, \ldots, u_{N-1})$.  

State includes Frenet/path states and dynamic states:  
$x_k = (\psi_k, e_{y,k}, s_{x,k}, V_k, \beta_k, r_k, \delta_k, \ldots)$,  
with $\delta_{k+1} = \delta_k + \Delta t \, \dot{\delta}_k$ capturing steering-rate control (as in the baseline).  

Dynamics constraints (discretized):  
$x_{k+1} - \Phi_{\Delta t}(x_k, u_k) = 0,$  
$k = 0, \ldots, N - 1,$  

where $\Phi_{\Delta t}$ is a numerical integrator applied to the continuous-time model $\dot{x} = f(x, u)$. The baseline uses the single-track dynamics (Eq. 1–2) and tire models as force closures.  

Stage cost for drifting equilibrium tracking (control-oriented):  

$$
\ell_k (x_k, u_k ; x_k^{eq}) =
\| \beta_k - \beta_k^{eq} \|{Q\beta}^2 +
\| r_k - r_k^{eq} \|_{Q_r}^2 +
\| e_{y,k} \|_{Q_y}^2 +
\| \dot{\delta}k \|{R_{\dot{\delta}}}^2 +
\| T_{R,k} - T_{R,k}^{eq} \|_{R_T}^2 + \cdots
$$

and terminal cost  

$$
\ell_N (x_N ; x_N^{eq}) = \| x_N - x_N^{eq} \|_{P}^2
$$

where $x^{eq}$ is provided by equilibrium maps and possibly modified by outer-loop curvature adjustment.  

Inequality constraints encode actuator-and-safety bounds:  
$g(x_k, u_k) \leq 0$  

(e.g., steering angle/rate bounds, torque bounds, slip bounds, maybe stability margins).


The key novelty proposed in this project to implement an SQP- Real Time iteration(RTI) solver inside JAX, as a compiled differentiable program and augment with vectorized multi start hybrid Gauss-Newton/HPV techniques. The idea is to retain the proven controls structure of multiple shooting and SQP while reaping the differentiable programming benefits.  Pure first order methors(SGD/ADAM) are attrative for compilation nad simplicity but have limitation with the stiff constraints and ill-conditioning in NMPC.  Differentiable programing provides efficient access to Gauss-Newton and inverse Hessian vector products  via (CG on HPVs), enabling quasi-second order behaviour without full Hessians[4]. 

Therefore, Differentiable Gauss-Newton SQP step structure can be implemented as : 

At iterate $w$, define linearization:  

$$
c(w) = 0,
$$

$$
c(w + \Delta w) \approx c(w) + J_c(w)\Delta w
$$

and approximate Lagrangian Hessian $H$ via Gauss–Newton where cost is sum-of-squares (as in tracking):  

$$
H \approx J_\phi^T W J_\phi + \lambda I,
$$

with $\phi(w)$ the stacked residual vector, and $\lambda$ a Levenberg–Marquardt damping term.  

The KKT system becomes banded due to horizon structure, allowing linear-time Riccati-like solves or sparse factorization.  

JAX advantage: obtain Jacobian-vector products (JVPs), vector-Jacobian products (VJPs), and Gauss–Newton vector products efficiently; compile the whole step including linear algebra and rollout.



Constraint handling: smooth barrier + augmented Lagrangian hybrid  

A central design challenge is reconciling differentiability with hard constraints. The proposed strategy is a hybrid:  

1. Hard box constraints (steering, torque) handled by projection or clipping on unconstrained variables (which is piecewise differentiable).  

2. General inequality constraints handled by a differentiable augmented Lagrangian:  

$$
\mathcal{L}_\rho (w, \lambda) = J(w) + \lambda^T \, \text{softplus}(g(w)) + \frac{\rho}{2} \left\| \text{softplus}(g(w)) \right\|^2
$$

where softplus smooths the hinge.  

3. Optional log-barrier for interior feasibility in early iterations:  

$$
J(w) - \mu \sum_i \log(-g_i(w))
$$

Differentiable programming emphasizes program smoothing as a core technique for branchy/non-smooth constructs. This hybrid provides a tunable trade-off: hard feasibility vs. smooth optimization and better gradient behavior.


Algorithmic structure in JAX: explicit pseudocode  
 
## Flowchart 
```{mermaid}
A[Offline: equilibrium map generation] --> B[Online: select equilibrium via curvature + path error]
B --> C[Warm-start: shift previous solution + equilibrium-consistent init]
C --> D[Compiled rollout: x_{0:N}(u) via scan]
D --> E[Objective + constraints residuals]
E --> F[Derivative engine: grad/JVP/VJP/HVP]
F --> G[Structured solve: GN-SQP / primal-dual step]
G --> H[Apply first control input]
H --> I[Vehicle + estimator update]
I --> B
E --> J[Validation hooks: scenario/falsification tests]
J --> A
```


## Algorithm: VM-GN-RTI (Vectorized Multi-start Gauss–Newton Real-Time Iteration)  

(“Real-time iteration” aligns with the MPC idea that exact solves are unnecessary; one or few SQP-like steps per sample can suffice.)  

1. Inputs: current estimate $x_t$, equilibrium reference $x_t^{eq}$, previous solution $u^{prev}$, candidate set size $M$.  

2. Warm-start generation: create $M$ candidate control sequences $u^{(i)}$ by perturbing $u^{prev}$ and/or using equilibrium-consistent templates (e.g., torque ramp + countersteer pattern).  

3. Vectorized rollout: compute $x_{0:N}^{(i)} = \text{rollout}(x_t, u^{(i)})$ for all $i$ in parallel (vmap).  

4. Vectorized cost+constraints: compute residuals $\phi^{(i)}$, constraints $g^{(i)}$.  

5. Differentiable update: for each $i$, compute GN step using Gauss–Newton vector products and a structured linear solve; apply $1$–$K$ iterations (with small $K$, e.g., $1$–$3$).  

6. Selection: choose the best feasible (or lowest augmented Lagrangian) candidate.  

7. Apply: send $u_0^\star$ to actuators; shift horizon.

## Computational efficiency, complexity, and bottlenecks  


  

The baseline drifting NMPC uses SQP at 50 Hz with finite horizon $N = 25$. SQP workflows typically involve:  

• Evaluate dynamics and residuals over horizon.  
• Build linearizations and (approximate) Hessians.  
• Solve a QP (banded/sparse structure).  
• Possibly line search/trust region.  

Real-time feasibility relies on small iteration counts and strong warm-starting (RTI-style behavior), consistent with MPC numerical guidance that exact solutions are unnecessary.  

The reason JAX based solver can be faster ..., 
1. End-to-end compilation reduces overhead. JAX explicitly compiles pure functions with XLA; compilation and autodiff are composable.  

2. Vectorization converts “many candidates” into one kernel. vmap pushes batch loops down to primitives, enabling efficient multi-start evaluation and Jacobian-like computations.  

3. Derivative efficiency with explicit trade-offs. Reverse-mode gradients are efficient for scalar objectives, but memory grows with horizon length; checkpointing manages this trade-off.  

4. Gauss–Newton vector products reduce second-order cost. GNVPs provide curvature information without full Hessians, requiring a small number of passes through the rollout.  


  



The novelty could be algorithmic, which are as follows :  

• Compiled solver step: treat “one RTI step” itself as a compiled numerical kernel, including rollouts, residual construction, and structured linear algebra—minimizing Python overhead and enabling fusion.  

• Vectorized multi-start + selection: use vmap to explore nonconvex basins cheaply in parallel, turning nonconvexity mitigation into a first-class design element.  

• Gauss–Newton/HVP blend: use Gauss–Newton where tracking is least-squares, and HVP-based CG where needed (e.g., for barrier-augmented parts), avoiding dense Hessians while retaining curvature information.  

• Checkpointed differentiation through horizon: explicitly manage memory using checkpointing, crucial for long horizons or multi-start batches.  

## Computational efficiency and validation methodology  

Computational efficiency: structured argument, complexity, and bottlenecks  

A credible efficiency claim must tie directly to the operations NMPC repeatedly performs.  

Baseline SQP cost profile  

The baseline drifting NMPC uses SQP at 50 Hz with finite horizon $N = 25$. SQP workflows typically involve:  

• Evaluate dynamics and residuals over horizon.  
• Build linearizations and (approximate) Hessians.  
• Solve a QP (banded/sparse structure).  
• Possibly line search/trust region.  

Real-time feasibility relies on small iteration counts and strong warm-starting (RTI-style behavior), consistent with MPC numerical guidance that exact solutions are unnecessary.  



## Robustness, numerical stability, and failure modes  

A drifting NMPC solver must be robust to:  

• Ill-conditioning near tire saturation.  
• Poor initial guesses during drift entry/transition.  
• Model mismatch (friction changes, unmodeled load transfer, actuator lag).  

The baseline explicitly observes that measurement noise and friction coefficient variations (~0.05 in their measured proving-ground context) affect stability and equilibrium tracking.  

Proposed stability safeguards:  

1. Damped GN steps (Levenberg–Marquardt). Guarantees descent directions for sufficiently large damping, mitigating divergence in nonconvex regions.  

2. Trust-region on control increments $\Delta u$ to protect actuator feasibility and reduce sensitivity to linearization.  

3. Constraint softening + penalty schedule to avoid infeasibility blow-ups during drift entry (then harden constraints as equilibrium is approached).  

4. Semi-decentralized fallback modes: if one module (e.g., steering) saturates or communication fails, degrade gracefully to a decentralized policy that stabilizes $\beta$ and $r$ even if path tracking relaxes—mirroring the semi-decentralization philosophy of mixed centralized/decentralized regimes.  

## Validation: Integrating safety-critical validation into drifting NMPC development  

The Algorithms for Validation project frames validation as algorithmic search over failure cases for safety-critical systems, emphasizing broad coverage of validation problem formulations and algorithms.  

For drifting NMPC, validation must be regime-aware:  

• Drift entry and exit transitions.  
• Surface friction heterogeneity and step changes.  
• Parameter mismatch (tire stiffness, friction, inertia).  
• Actuator saturation events and delays.  

## Validation pipeline:  

1. Model-in-the-loop (MIL): randomized scenarios + adversarial perturbations to stress equilibrium capture and solver convergence.  

2. Software-in-the-loop (SIL): compiled JAX solver-in-loop with timing instrumentation, verifying real-time budgets and deterministic latencies.  

3. Hardware-in-the-loop (HIL): ECU-like deployment with realistic interfaces (CAN/FlexRay), explicitly addressing the baseline’s identified actuator-interface constraints.  

4. Falsification-oriented scenario search: Treat constraint violation or divergence as objective; use gradient-informed search (since the closed loop is differentiable in simulation) to generate “minimum-energy” adversarial scenarios.  




# Outline in Summary 

The project studies automated drifting as an extreme nonlinear control problem and proposes a new solver architecture: which is complied, differentiable, vectorized NMPC pipeline in JAX for drifting under tire saturation , unstable equilibira, actuator limits and tighet real time constraints.  As above, In the proposal I have explicitly framed drifting regime as the regime where linear or mildly nonlinear assumptions fail, and iterative Standard Qp  based MPC can become fragile, and where solver design itself become the contribution. Therefore, the purpose of the work is therefore to show that NMPC can be reformualted as a single differentiable program whose rollout, residual construction, derivative computation, and structured update are compiled together, while also incoporating multi-model prograbilistic fusion, semi-decentralized computation, and safety aware validation.  Since, JAX is built around a composable transforms such as VMAP< JIT< GRAD , and because memroy computation trafeoffs can be managed explicitly with chekcpointing. 



Intutively, the difficulty of automated drifting does not arise because the vehicle accidentally departs from nominal operation, but because the controller intentionally drives the vehicle into a regime where stable operation is no longer natural. In drifting, the objective is to maintain a large sideslip angle and exploit rear-tire saturation to achieve a desired motion, which means the controller must deliberately use aggressive steering and torque actions while simultaneously respecting hard steering-angle, steering-rate, drivetrain, and actuator constraints. Under these conditions, the system is highly nonlinear, strongly coupled, and often close to unstable equilibria, so the controller is not simply correcting small tracking errors around a safe operating point; it is continuously balancing the vehicle at the boundary between controlled motion and spin-out. Recent drifting-control work reflects exactly this challenge by formulating the task as stabilizing and maintaining a vehicle at high-sideslip conditions rather than as ordinary path tracking. This is why this proposal wants to test for a compiled differentiable NMPC architecture where,  the controller must repeatedly predict many possible future trajectories of the nonlinear vehicle, evaluate which ones satisfy the drifting objective and physical limits, compute structured derivatives efficiently, and update the control sequence fast enough for real-time execution, while an additional safety layer can intervene if the nominal optimizer becomes too aggressive. In that sense, the problem is complex precisely because we want the vehicle to drift on purpose, but we still require that this intentionally unstable behavior remain physically feasible, computationally tractable, and safety-aware at every control step.

### Central Objective: 

Is a JAX - native differentiable optimization stack the best technical basis for real time driftign NMPC? Is it possible to retain the proven control structure of multiple shooting, SQP, and RTI, while replacing the classical “solve one NLP by repeated external calls” viewpoint with a compiled computational graph that supports fast derivatives, batched multi-start exploration, structured linear algebra, and learned model adaptation? Therefore, response to four coupled challenges in drifting: strong tire-force nonlinearity and coupling, unstable high-sideslip equilibria, tight steering/torque/rate limits, and the fact that exact solves are often less useful than fast structured approximate ones in receding horizon control.Therefore, Real-time nonlinear MPC remains the benchmark alternative; acados, for example, is explicitly designed for fast embedded nonlinear MPC and supports RTI and advanced-step RTI, so the proposed JAX- based stack architecture must be judged against that standard solver..., 

### Objective
The objective is to demonstrate a solver-level innovation: A Vectorized Multi-start Gauss–Newton Real-Time Iteration (VM-GN-RTI) scheme in which candidate control sequences are generated from shifted warm starts and equilibrium-consistent templates, rolled out in parallel, evaluated with differentiable costs and constraints, updated by Gauss–Newton / Hessian-vector-product structure, and then filtered through safety-aware logic before the first action is applied. The intended contribution has four layers:

1. Algorithmic: compiled RTI as a single differentiable kernel.
2. Numerical: multi-start plus GN/HVP structure to mitigate nonconvexity without dense Hessians.
3. Modeling: multi-model probabilistic fusion for severe model mismatch.
4. Validation: regime-aware falsification and solver-failure analysis as part of the method, not an afterthough

###
The online loop should be written as a compact outline:
Offline: generate equilibrium maps, nominal drift references, and possibly candidate model families.
Online Step 1: estimate current state and local curvature/path context.
Step 2: generate warm-start candidates by shifting the previous optimal sequence and perturbing it with equilibrium-consistent templates such as torque ramps and countersteer patterns.
Step 3: roll out all candidates with a compiled scan-based integrator.
Step 4: accumulate objective residuals and inequality residuals.
Step 5: compute derivatives using grad, JVPs, VJPs, and where needed Gauss–Newton vector products or HVP-based conjugate-gradient structure.
Step 6: solve the structured GN-SQP / primal-dual subproblem
Step 7  Select the best feasible augmented- Larangian candidate 
Step 8: Pass the control first contrl through a safety filter, apply it, shift the horizon and repeat


### Constraint handling should be presented in two layers. 

The optimization layer uses a hybrid of clipping/projection for hard box constraints, smooth augmented-Lagrangian terms for general inequalities, and optional log barriers for early interior feasibility; this keeps the inner problem sufficiently smooth for differentiation and GN-style updates. The safety layer sits on top of the nominal optimizer and enforces track, obstacle, and state-envelope safety through Control Barrier Functions. This distinction matters: penalties and barriers help optimization, but they are not the same as a runtime safety mechanism. The proposal already states this hybrid philosophy; CBFpy is the JAX-native tool that fits the second layer because it is built specifically for CBF/CLF construction with JIT, XLA and autodiff. 

### Libaries and Roles: 

1. Diffrax: primary rollout engine for the nonlinear vehicle model; use for differentiable integration, event handling, and adjoint-based gradients. 
2. Lineax: linear solves and linear least squares for KKT systems, GN normal equations, and operator-based Jacobian actions. 
3. Optimistix: nonlinear least squares, root finding, and implicit-differentiation-friendly solver blocks; use for LM, Dogleg, or custom subsolves. 
4. Equinox: lightweight model definitions for learned residuals, model-fusion weights, damping schedules, or cost shapers; ideal because it is explicitly “not a framework” and stays close to raw JAX. It is important  for learnable pieces.
5. Optax: parameter optimization for the learned modules; useful for tuning residual models, uncertainty heads, and weighting networks. It is central for training but not for control subproblems.
6. CBFpy: Runtime safety filter or CLF-CBF wrapper around the nominal drifting controller. It is central for safety but secondary to the nominal NMPC itself.
7. CVXPYLayers: optional differentiable convex sublayer for actuator allocation, convex smoothing, or projection into local convex safety tubes. It is useful only where a true convex subproblem exists.
8. Trajax: useful as a prototype differentiable optimal-control baseline or warm-start generator, but it is archived and therefore should not be the long-term core.
9. MJX: secondary differentiable physics backend for broader physics validation; more relevant than Brax because MJX is the maintained successor to Brax’s generalized pipeline.
10. Brax: mainly auxiliary RL/training infrastructure now; not the primary simulator for this proposal.
11. Waymax: strong JAX traffic simulator, but not a good primary drift simulator because it is aimed at object-level autonomous-driving behavior research and uses a default kinematic bicycle abstraction. Keep it only for downstream multi-agent road-scene validation
12. Dynamax : It provides a probabilstic state space models and inference learning in JAX. Without this addition, the fusion layer must be hand-built inside equinox, which is possible but weaker. (This comment is not yet validated , found form the use comment)























































[1) REF: Kouvaritakis, Basil, and Mark Cannon. "Model predictive control." Switzerland: Springer International Publishing 38.13-56 (2016): 7. Chapter 1].  
[2) Mayne, David. "Nonlinear model predictive control: Challenges and opportunities." Nonlinear model predictive control (2000): 23-44.]  
[[3] Rawlings, James B., David Q. Mayne, and Moritz M. Diehl. "Model predictive control: theory, computation, and design." (No Title) (2020).]  
[[4] Blondel, Mathieu, and Vincent Roulet. "The elements of differentiable programming." arXiv preprint arXiv:2403.14606 (2024).]]  
[[5] Bradbury, James, et al. "JAX: Composable transformations of Python+ NumPy programs, version 0.3.13, 2018." Online at http://github.com/jax-ml/jax (accessed on October 31, 2025) (2018).]]  
[[6] Houska, Boris, and Yuning Jiang. "Distributed optimization and control with ALADIN." Recent Advances in Model Predictive Control: Theory, Algorithms, and Applications. Cham: Springer International Publishing, 2021. 135-163.]]  
[[7] Schulze Darup, Moritz, and Gerrit Book. "On closed-loop dynamics of ADMM-based MPC." Recent Advances in Model Predictive Control: Theory, Algorithms, and Applications. Cham: Springer International Publishing, 2021. 107-134.]  
[[8.] Al-Husseini, Mahdi, Mykel J. Kochenderfer, and Kyle H. Wray. "A Semi-Decentralized Approach to Multiagent Control." arXiv preprint arXiv:2603.11802 (2026).]  
[[9.]] Meijer, Stan, Alberto Bertipaglia, and Barys Shyrokau. "A nonlinear model predictive control for automated drifting with a standard passenger vehicle." 2024 IEEE International Conference on Advanced Intelligent Mechatronics (AIM). IEEE, 2024.  
[[10.]] Pitre, Ryan R., Vesselin P. Jilkov, and X. Rong Li. "A comparative study of multiple-model algorithms for maneuvering target tracking." Signal Processing, Sensor Fusion, and Target Recognition XIV. Vol. 5809. SPIE, 2005.

[[11.]] Chowdhri, Nishant, et al. "Integrated nonlinear model predictive control for automated driving." Control Engineering Practice 106 (2021): 104654.
[[12.]] Righetti, Giovanni, et al. "Drifting Maneuver Investigation via Phase Plane Analysis of Experimental Data." Advanced Vehicle Control Symposium. Cham: Springer Nature Switzerland, 2024.






In [2]:
import sys
import jax
import jax.numpy as jnp
print(jax.__version__)

0.9.0.1


In [4]:
!pip freeze

absl-py==2.4.0
aiofiles==25.1.0
anyio==4.12.1
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.4.0
asttokens==3.0.1
async-lru==2.2.0
attrs==25.4.0
babel==2.18.0
beautifulsoup4==4.14.3
bleach==6.3.0
certifi==2026.2.25
cffi==2.0.0
charset-normalizer==3.4.5
chex==0.1.91
colorama==0.4.6
comm==0.2.3
dataclasses-json==0.6.7
debugpy==1.8.20
decorator==5.2.1
defusedxml==0.7.1
dm-haiku==0.0.16
etils==1.13.0
executing==2.2.1
fastjsonschema==2.21.2
flax==0.12.4
fqdn==1.5.1
fsspec==2026.2.0
h11==0.16.0
httpcore==1.0.9
httpx==0.28.1
humanize==4.15.0
idna==3.11
importlib_resources==6.5.2
ipykernel==7.2.0
ipython==9.11.0
ipython_pygments_lexers==1.1.1
isoduration==20.11.0
jax==0.9.0.1
jaxlib==0.9.0.1
jaxtyping==0.3.9
jedi==0.19.2
Jinja2==3.1.6
jmp==0.0.4
json5==0.13.0
jsonpointer==3.0.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter-events==0.12.0
jupyter-lsp==2.3.0
jupyter_client==8.8.0
jupyter_core==5.9.1
jupyter_server==2.17.0
jupyter_server_terminals==0.5.4
jupyterlab==4.

In [6]:
os.listdir("Image")


['NLP_Drift.png']